In [0]:
import requests
import hashlib

from datetime import datetime, timezone, timedelta

pipeline_start_timestamp = datetime.now(timezone.utc)

PIPELINE_NAME = "bcb_sgs_selic"

print("Pipeline start:", pipeline_start_timestamp)

In [0]:
BCB_BASE_URL = "https://api.bcb.gov.br/dados/serie"
SOURCE_SYSTEM = "BCB_SGS"
SERIES_CODE = "432"
LOOKBACK_DAYS = 30
REQUEST_TIMEOUT_SECONDS = 30

In [0]:
execution_id = dbutils.jobs.taskValues.get(
    taskKey="INITIALIZE_MONITORING",
    key="execution_id"
)

step_start_timestamp = datetime.now(timezone.utc)

print("Pipeline Execution ID:", execution_id)

spark.sql(f"""
UPDATE workspace.brazilian_economic_monitoring.pipeline_step_execution
SET
    status = 'RUNNING',
    start_timestamp = CURRENT_TIMESTAMP()
WHERE execution_id = '{execution_id}'
  AND step_name = 'INGEST_BRONZE'
""")

print("INGEST_BRONZE status: RUNNING")

In [0]:
try:

    # ---------------------------------------------------------
    # 1. Define API request window
    # ---------------------------------------------------------

    end_date = datetime.now(timezone.utc).date()
    start_date = end_date - timedelta(days=LOOKBACK_DAYS)

    start_date_bcb = start_date.strftime("%d/%m/%Y")
    end_date_bcb = end_date.strftime("%d/%m/%Y")

    url = f"{BCB_BASE_URL}/bcdata.sgs.{SERIES_CODE}/dados"

    params = {
        "formato": "json",
        "dataInicial": start_date_bcb,
        "dataFinal": end_date_bcb
    }

    # ---------------------------------------------------------
    # 2. Extract data from BCB API
    # ---------------------------------------------------------

    response = requests.get(
        url,
        params=params,
        timeout=REQUEST_TIMEOUT_SECONDS
    )

    response.raise_for_status()

    records = response.json()

    print("Source:", SOURCE_SYSTEM)
    print("Series:", SERIES_CODE)
    print("Start date:", start_date_bcb)
    print("End date:", end_date_bcb)
    print("HTTP Status:", response.status_code)
    print("URL:", response.url)
    print("Records received:", len(records))
    print("First record:", records[0] if records else None)

    if not records:
        raise ValueError(
            f"No records returned by BCB SGS series {SERIES_CODE} "
            f"between {start_date_bcb} and {end_date_bcb}"
        )

    # ---------------------------------------------------------
    # 3. Prepare Bronze records
    # ---------------------------------------------------------

    ingestion_timestamp = datetime.now(timezone.utc)

    bronze_records = [
        (
            item["data"],
            item["valor"],
            SOURCE_SYSTEM,
            SERIES_CODE,
            ingestion_timestamp,
            ingestion_timestamp.date(),
            execution_id,
            hashlib.sha256(
                f"{SERIES_CODE}||{item['data']}||{item['valor']}".encode("utf-8")
            ).hexdigest()
        )
        for item in records
    ]

    columns = [
        "reference_date_raw",
        "value_raw",
        "source_system",
        "source_series_code",
        "ingestion_timestamp",
        "ingestion_date",
        "execution_id",
        "record_hash"
    ]

    df_bronze = spark.createDataFrame(
        bronze_records,
        schema=columns
    )

    df_bronze.createOrReplaceTempView(
        "vw_bcb_sgs_selic_bronze"
    )

    print("Records prepared:", len(bronze_records))

    # ---------------------------------------------------------
    # 4. Bronze volume before MERGE
    # ---------------------------------------------------------

    bronze_count_before = spark.sql("""
        SELECT COUNT(*) AS count
        FROM workspace.brazilian_economic_bronze.bcb_sgs_selic
    """).collect()[0]["count"]

    # ---------------------------------------------------------
    # 5. Idempotent Bronze MERGE
    # ---------------------------------------------------------

    spark.sql("""
        MERGE INTO workspace.brazilian_economic_bronze.bcb_sgs_selic AS target

        USING vw_bcb_sgs_selic_bronze AS source

        ON target.record_hash = source.record_hash

        WHEN NOT MATCHED THEN INSERT (
            reference_date_raw,
            value_raw,
            source_system,
            source_series_code,
            ingestion_timestamp,
            ingestion_date,
            execution_id,
            record_hash
        )

        VALUES (
            source.reference_date_raw,
            source.value_raw,
            source.source_system,
            source.source_series_code,
            source.ingestion_timestamp,
            source.ingestion_date,
            source.execution_id,
            source.record_hash
        )
    """)

    # ---------------------------------------------------------
    # 6. Calculate metrics
    # ---------------------------------------------------------

    bronze_count_after = spark.sql("""
        SELECT COUNT(*) AS count
        FROM workspace.brazilian_economic_bronze.bcb_sgs_selic
    """).collect()[0]["count"]

    records_inserted_bronze = (
        bronze_count_after - bronze_count_before
    )

    print("Records received:", len(records))
    print("Bronze count before:", bronze_count_before)
    print("Bronze count after:", bronze_count_after)
    print(
        "Records inserted into Bronze:",
        records_inserted_bronze
    )

    # ---------------------------------------------------------
    # 7. Monitoring SUCCESS
    # ---------------------------------------------------------

    step_end_timestamp = datetime.now(timezone.utc)

    duration_seconds = int(
        (step_end_timestamp - step_start_timestamp).total_seconds()
    )

    spark.sql(f"""
        UPDATE workspace.brazilian_economic_monitoring.pipeline_step_execution

        SET
            status = 'SUCCESS',
            end_timestamp = CURRENT_TIMESTAMP(),
            duration_seconds = {duration_seconds},
            records_read = {len(records)},
            records_inserted = {records_inserted_bronze},
            records_updated = 0,
            records_output = {bronze_count_after},
            failed_checks = NULL,
            error_message = NULL

        WHERE execution_id = '{execution_id}'
          AND step_name = 'INGEST_BRONZE'
    """)

    print("INGEST_BRONZE status: SUCCESS")
    print("Duration seconds:", duration_seconds)

except Exception as e:

    # ---------------------------------------------------------
    # Monitoring FAILED
    # ---------------------------------------------------------

    step_end_timestamp = datetime.now(timezone.utc)

    duration_seconds = int(
        (step_end_timestamp - step_start_timestamp).total_seconds()
    )

    error_message = str(e).replace("'", "''")[:4000]

    spark.sql(f"""
        UPDATE workspace.brazilian_economic_monitoring.pipeline_step_execution

        SET
            status = 'FAILED',
            end_timestamp = CURRENT_TIMESTAMP(),
            duration_seconds = {duration_seconds},
            error_message = '{error_message}'

        WHERE execution_id = '{execution_id}'
          AND step_name = 'INGEST_BRONZE'
    """)

    print("INGEST_BRONZE status: FAILED")
    print("Error:", str(e))

    raise